# Semana 2 — Transformación del Dataset para Machine Learning

**Objetivo:** Transformar el dataset original (formato largo, 2000–2022) a un formato utilizable para regresión.

**Problema a resolver:** Estimar el porcentaje de uso de Internet por país, año y grupo etario.

---

**Archivo principal:** `02Semana2Transformacion.ipynb`
**Dataset transformado:** `outputs/datos_transformados_semana2.csv`

## 1. Diseño del Dataset Transformado

### 1. Análisis del Formato Original

El dataset original está en formato **largo**, donde cada fila representa una observación de país, año y grupo etario. Esta estructura es ideal para exploración visual pero **no es suficiente para regresión** por las siguientes razones:

- **Carencia de variables temporales explícitas:** El año es una dimensión de la tabla, no una variable numérica que capture tendencia.
- **Sin estructura tabular directa:** Para regresión supervisa necesitamos variables independientes en columnas y variable objetivo aislada.
- **Columnas redundantes:** `indicator`, `unit`, `notes_ids`, `source_id` tienen valores fijos o vacíos sin aportar señal predictiva.
- **Panel desbalanceado:** Requiere limpieza (valores faltantes, período confiable desde 2016).

**Recomendación de Semana 1:** Pivotaje a ancho + codificación categórica + variables temporales.

### 2. Clasificación de Columnas del Dataset Original

| Columna | Tipo | Decisión | Justificación |
|---------|------|----------|---------------|
| `indicator` | string (fijo) | **ELIMINAR** | Valor constante; sin variabilidad para ML |
| `País__ESTANDAR` | string | **CONSERVAR** | Dimensión clave del problema; diferencia importante entre países |
| `Grupos etarios Uso Internet` | string | **CONSERVAR** (sin "Total") | Dimensión clave; modelar por grupo etario específico |
| `Años__ESTANDAR` | numeric | **CONSERVAR + TRANSFORMAR** | Base temporal; crear `years_since_2016` para capturar tendencia |
| `value` | numeric (0–100) | **CONSERVAR** | Variable objetivo; porcentaje de usuarios de Internet |
| `unit` | string (fijo) | **ELIMINAR** | Valor constante; sin valor predictivo |
| `notes_ids` | string (vacío) | **ELIMINAR** | Mayormente faltante; sin valor para modelo |
| `source_id` | numeric (fijo) | **ELIMINAR** | Identificador de fuente; no añade información analítica |

### 3. Propuesta de Estructura Final del Dataset

#### Unidad de análisis
**Cada fila = una observación de país, año y grupo etario con su tasa de uso de Internet.**

**Período:** 2016–2022 (confiabilidad post-2016, Semana 1)  
**Filtros:** Sin grupo "Total"; solo grupos etarios específicos

#### Columnas del CSV Transformado

| Columna | Tipo | Rango/Valores | Descripción |
|---------|------|---------------|-------------|
| `pais` | string | 13 países únicos | Nombre estandarizado del país (de `País__ESTANDAR`) |
| `año` | int | 2016–2022 | Año del registro (de `Años__ESTANDAR`, filtrado) |
| `years_since_2016` | int | 0–6 | Años transcurridos desde 2016 (variable temporal derivada) |
| `grupo_etario` | string | 5 grupos | Grupo de edad (de `Grupos etarios Uso Internet`, sin "Total") |
| `porcentaje_internet` | float | 0.0–100.0 | Porcentaje de usuarios de Internet (**variable objetivo**) |

#### Tamaño esperado
- Combinaciones posibles: 13 países × 7 años × 5 grupos = **455 filas** (sin valores faltantes)
- Estructura resultante: **455 × 5** (455 observaciones, 5 columnas)
- Formato: tabular directo, listo para regresión supervisada

### 4. Justificación del Diseño Propuesto

#### Por qué esta estructura es mejor que el original

1. **Tabular y limpia**
   - Cada fila es independiente (observación completa)
   - Todas las variables en columnas, lista para modelos de ML
   - Sin ruido de columnas redundantes

2. **Captura de tendencia temporal**
   - `years_since_2016` es una variable numérica derivada que permite al modelo aprender aceleración/desaceleración
   - Facilita la captura de dinámicas entre 2016 y 2022

3. **Alineación con el problema**
   - Problema: "Regresión para estimar porcentaje por país, año y grupo"
   - Diseño: cada fila tiene país, año (+ tendencia), grupo y objetivo
   - Estructura 1-a-1 con los requisitos del problema

4. **Período confiable**
   - Solo 2016–2022: datos post-2016 tienen mayor cobertura y consistencia (Semana 1)
   - Reduce ruido de períodos con baja confiabilidad

5. **Sin variables redundantes**
   - Eliminadas columnas con valores fijos (`indicator`, `unit`, `source_id`)
   - Eliminada columna mayormente vacía (`notes_ids`)
   - Reducción de dimensionalidad para mejor generalización del modelo

#### Decisiones clave registradas

| Decisión | Valor | Justificación |
|----------|-------|---------------|
| Unidad de análisis | país-año-grupo | Estructura requerida para regresión supervisada |
| Período | 2016–2022 | Confiabilidad post-2016 (Semana 1) |
| Filtro grupo "Total" | Eliminar | Solo grupos etarios específicos para modelar variabilidad por edad |
| Variables derivadas | `years_since_2016` | Captura tendencia temporal numérica |
| Columnas eliminadas | `indicator`, `unit`, `notes_ids`, `source_id` | Sin valor predictivo |

## 2. Transformación del Dataset

In [1]:
import pandas as pd
import numpy as np
import os

df_original = pd.read_csv('../data/datos.csv', sep=';', encoding='latin1')
print(f"Dataset original: {df_original.shape[0]} filas x {df_original.shape[1]} columnas")
print(f"Columnas: {list(df_original.columns)}")
df_original.head()

Dataset original: 870 filas x 8 columnas
Columnas: ['indicator', 'País__ESTANDAR', 'Grupos etarios Uso Internet', 'Años__ESTANDAR', 'value', 'unit', 'notes_ids', 'source_id']


,indicator,País__ESTANDAR,Grupos etarios Uso Internet,Años__ESTANDAR,value,unit,notes_ids,source_id
0,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2016,76,Porcentaje sobre el total de personas en cada ...,NaN,9353
1,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2017,76,Porcentaje sobre el total de personas en cada ...,NaN,9353
2,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2018,79,Porcentaje sobre el total de personas en cada ...,NaN,9353
3,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2019,79,Porcentaje sobre el total de personas en cada ...,NaN,9353
4,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2020,88,Porcentaje sobre el total de personas en cada ...,NaN,9353


In [2]:
col_pais = df_original.columns[1]   # País__ESTANDAR
col_anios = df_original.columns[3]  # Años__ESTANDAR
col_grupo = 'Grupos etarios Uso Internet'

# Filtrar período 2016–2022 y excluir grupo "Total"
df_filtrado = df_original[
    (df_original[col_anios] >= 2016) &
    (df_original[col_anios] <= 2022) &
    (df_original[col_grupo] != 'Total')
].copy()

print(f"Filas después de filtrar: {df_filtrado.shape[0]}")
print(f"Países únicos: {df_filtrado[col_pais].nunique()}")
print(f"Años únicos: {sorted(df_filtrado[col_anios].unique())}")
print(f"Grupos etarios: {list(df_filtrado[col_grupo].unique())}")

Filas después de filtrar: 340
Países únicos: 13
Años únicos: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Grupos etarios: ['edad de medicion a 17 años', '18 a 25 años de edad', '26 a 50 años de edad', '51 a 65 años', '66 años en adelante']


In [3]:
# Selección, renombrado y creación de variable temporal
df_transformado = df_filtrado.rename(columns={
    col_pais: 'pais',
    col_anios: 'año',
    col_grupo: 'grupo_etario',
    'value': 'porcentaje_internet'
})[['pais', 'año', 'grupo_etario', 'porcentaje_internet']].copy()

df_transformado['years_since_2016'] = df_transformado['año'] - 2016
df_transformado['porcentaje_internet'] = df_transformado['porcentaje_internet'].astype(float)

# Codificación de variables dummy para grupo_etario
dummies = pd.get_dummies(df_transformado['grupo_etario'], prefix='grupo', dtype=int)
df_ml = pd.concat([df_transformado, dummies], axis=1)

print(f"Dataset transformado: {df_ml.shape[0]} filas x {df_ml.shape[1]} columnas")
print(f"Columnas finales: {list(df_ml.columns)}")
df_ml.head()

Dataset transformado: 340 filas x 10 columnas
Columnas finales: ['pais', 'año', 'grupo_etario', 'porcentaje_internet', 'years_since_2016', 'grupo_18 a 25 años de edad', 'grupo_26 a 50 años de edad', 'grupo_51 a 65 años', 'grupo_66 años en adelante', 'grupo_edad de medicion a 17 años']


,pais,año,grupo_etario,porcentaje_internet,years_since_2016,grupo_18 a 25 años de edad,grupo_26 a 50 años de edad,grupo_51 a 65 años,grupo_66 años en adelante,grupo_edad de medicion a 17 años
0,Argentina,2016,edad de medicion a 17 años,76.0,0,0,0,0,0,1
1,Argentina,2017,edad de medicion a 17 años,76.0,1,0,0,0,0,1
2,Argentina,2018,edad de medicion a 17 años,79.0,2,0,0,0,0,1
3,Argentina,2019,edad de medicion a 17 años,79.0,3,0,0,0,0,1
4,Argentina,2020,edad de medicion a 17 años,88.0,4,0,0,0,0,1


In [4]:
# Validación rápida antes de exportar
print("Duplicados:", df_ml.duplicated().sum())
print("Nulos explícitos:\n", df_ml.isnull().sum().to_string())
print("\nRango porcentaje_internet:", df_ml['porcentaje_internet'].min(), "–", df_ml['porcentaje_internet'].max())
print("Países:", sorted(df_ml['pais'].unique()))

Duplicados: 0
Nulos explícitos:
 pais                                0
año                                 0
grupo_etario                        0
porcentaje_internet                 0
years_since_2016                    0
grupo_18 a 25 años de edad          0
grupo_26 a 50 años de edad          0
grupo_51 a 65 años                  0
grupo_66 años en adelante           0
grupo_edad de medicion a 17 años    0

Rango porcentaje_internet: 3.0 – 97.0
Países: ['Argentina', 'Bolivia (Estado Plurinacional de)', 'Chile', 'Colombia', 'Costa Rica', 'Ecuador', 'El Salvador', 'Honduras', 'México', 'Panamá', 'Paraguay', 'Perú', 'Uruguay']


In [5]:
os.makedirs('../outputs', exist_ok=True)
df_ml.to_csv('../outputs/datos_transformados_semana2.csv', index=False, encoding='utf-8')
print("CSV exportado: outputs/datos_transformados_semana2.csv")

CSV exportado: outputs/datos_transformados_semana2.csv


## 3. Validación de Calidad del Dataset Final

Se revisan valores faltantes (explícitos e implícitos), duplicados, rangos, tipos de dato y coherencia general del dataset transformado.

### 3.1 Valores Faltantes Explícitos: Original vs. Transformado

In [6]:
# Nulos explícitos en el dataset original
print("=== Nulos en dataset original ===")
print(df_original.isnull().sum().to_string())

print("\n=== Nulos en dataset transformado (df_ml) ===")
print(df_ml.isnull().sum().to_string())

=== Nulos en dataset original ===
indicator                        0
País__ESTANDAR                   0
Grupos etarios Uso Internet      0
Años__ESTANDAR                   0
value                            0
unit                             0
notes_ids                      870
source_id                        0

=== Nulos en dataset transformado (df_ml) ===
pais                                0
año                                 0
grupo_etario                        0
porcentaje_internet                 0
years_since_2016                    0
grupo_18 a 25 años de edad          0
grupo_26 a 50 años de edad          0
grupo_51 a 65 años                  0
grupo_66 años en adelante           0
grupo_edad de medicion a 17 años    0


### 3.2 Valores Faltantes Implícitos (Combinaciones Ausentes)

El dataset transformado no tiene nulos explícitos, pero al verificar todas las combinaciones posibles de país × año × grupo etario se detectan **combinaciones faltantes** (filas inexistentes). Estas representan observaciones donde no hay datos disponibles y no generan una fila con NaN, sino que directamente no existen.

In [7]:
# Construir índice completo de todas las combinaciones posibles
paises = df_transformado['pais'].unique()
anios = range(2016, 2023)
grupos = df_transformado['grupo_etario'].unique()

indice_completo = pd.MultiIndex.from_product(
    [paises, anios, grupos],
    names=['pais', 'año', 'grupo_etario']
)
total_posible = len(indice_completo)

indice_existente = df_transformado.set_index(['pais', 'año', 'grupo_etario']).index
faltantes = indice_completo.difference(indice_existente)
faltantes_df = pd.DataFrame(list(faltantes), columns=['pais', 'año', 'grupo_etario'])

print(f"Combinaciones posibles: {total_posible}")
print(f"Combinaciones presentes: {len(df_transformado)}")
print(f"Combinaciones faltantes: {len(faltantes_df)} ({len(faltantes_df)/total_posible*100:.1f}%)")
print("\nFaltantes por país:")
print(faltantes_df.groupby('pais').size().sort_values(ascending=False).to_string())
print("\nFaltantes por año:")
print(faltantes_df.groupby('año').size().to_string())

Combinaciones posibles: 455
Combinaciones presentes: 340
Combinaciones faltantes: 115 (25.3%)

Faltantes por país:
pais
Chile                                30
Honduras                             20
El Salvador                          15
Ecuador                              15
Uruguay                              15
Panamá                               10
Colombia                              5
Bolivia (Estado Plurinacional de)     5

Faltantes por año:
año
2016     5
2017    10
2018     5
2019     5
2020    30
2021    30
2022    30


In [8]:
# Matriz de cobertura: filas por país y año
cobertura = df_transformado.groupby(['pais', 'año']).size().unstack(fill_value=0)
print("Cobertura por país y año (número de grupos etarios con datos):")
print(cobertura.to_string())

Cobertura por país y año (número de grupos etarios con datos):
año                                2016  2017  2018  2019  2020  2021  2022
pais                                                                       
Argentina                             5     5     5     5     5     5     5
Bolivia (Estado Plurinacional de)     5     5     5     5     5     5     0
Chile                                 0     5     0     0     0     0     0
Colombia                              5     0     5     5     5     5     5
Costa Rica                            5     5     5     5     5     5     5
Ecuador                               5     5     5     5     0     0     0
El Salvador                           5     5     5     5     0     0     0
Honduras                              5     0     5     5     0     0     0
México                                5     5     5     5     5     5     5
Panamá                                5     5     5     5     0     0     5
Paraguay                 

### 3.3 Duplicados

In [9]:
dup_filas = df_ml.duplicated().sum()
dup_clave = df_ml.duplicated(['pais', 'año', 'grupo_etario']).sum()

print(f"Filas duplicadas (completas): {dup_filas}")
print(f"Duplicados por clave (pais, año, grupo_etario): {dup_clave}")

Filas duplicadas (completas): 0
Duplicados por clave (pais, año, grupo_etario): 0


### 3.4 Valores Fuera de Rango e Inconsistencias

In [10]:
fuera_rango = ((df_ml['porcentaje_internet'] < 0) | (df_ml['porcentaje_internet'] > 100)).sum()
print(f"Valores fuera de rango [0, 100] en porcentaje_internet: {fuera_rango}")
print(f"Rango observado: {df_ml['porcentaje_internet'].min():.1f} – {df_ml['porcentaje_internet'].max():.1f}")
print(f"\nEstadísticas descriptivas de porcentaje_internet:")
print(df_ml['porcentaje_internet'].describe().to_string())

anios_invalidos = (~df_ml['año'].between(2016, 2022)).sum()
ysr_invalidos = (~df_ml['years_since_2016'].between(0, 6)).sum()
print(f"\nAños fuera de 2016–2022: {anios_invalidos}")
print(f"years_since_2016 fuera de 0–6: {ysr_invalidos}")
print(f"\nGrupos etarios únicos:\n{df_ml['grupo_etario'].value_counts().to_string()}")

Valores fuera de rango [0, 100] en porcentaje_internet: 0
Rango observado: 3.0 – 97.0

Estadísticas descriptivas de porcentaje_internet:
count    340.000000
mean      58.911765
std       25.896399
min        3.000000
25%       39.000000
50%       64.000000
75%       80.250000
max       97.000000

Años fuera de 2016–2022: 0
years_since_2016 fuera de 0–6: 0

Grupos etarios únicos:
grupo_etario
edad de medicion a 17 años    68
18 a 25 años de edad          68
26 a 50 años de edad          68
51 a 65 años                  68
66 años en adelante           68


### 3.5 Tipos de Dato

In [11]:
print("Tipos de dato en el dataset transformado:")
print(df_ml.dtypes.to_string())
print("\nVerificación esperada:")
esperados = {
    'pais': 'object', 'año': 'int64', 'grupo_etario': 'object',
    'porcentaje_internet': 'float64', 'years_since_2016': 'int64'
}
for col, tipo_esp in esperados.items():
    tipo_real = str(df_ml[col].dtype)
    estado = "OK" if tipo_real == tipo_esp else f"REVISAR (es {tipo_real})"
    print(f"  {col}: esperado {tipo_esp} — {estado}")

Tipos de dato en el dataset transformado:
pais                                    str
año                                   int64
grupo_etario                            str
porcentaje_internet                 float64
years_since_2016                      int64
grupo_18 a 25 años de edad            int64
grupo_26 a 50 años de edad            int64
grupo_51 a 65 años                    int64
grupo_66 años en adelante             int64
grupo_edad de medicion a 17 años      int64

Verificación esperada:
  pais: esperado object — REVISAR (es str)
  año: esperado int64 — OK
  grupo_etario: esperado object — REVISAR (es str)
  porcentaje_internet: esperado float64 — OK
  years_since_2016: esperado int64 — OK


### 3.6 Coherencia del CSV Final

In [12]:
df_csv = pd.read_csv('../outputs/datos_transformados_semana2.csv', encoding='utf-8')
print(f"Filas en CSV: {df_csv.shape[0]}")
print(f"Columnas en CSV: {df_csv.shape[1]}")
print(f"Columnas: {list(df_csv.columns)}")
print(f"\nNulos en CSV:\n{df_csv.isnull().sum().to_string()}")
print(f"\nPrimeras filas:")
df_csv.head()

Filas en CSV: 340
Columnas en CSV: 10
Columnas: ['pais', 'año', 'grupo_etario', 'porcentaje_internet', 'years_since_2016', 'grupo_18 a 25 años de edad', 'grupo_26 a 50 años de edad', 'grupo_51 a 65 años', 'grupo_66 años en adelante', 'grupo_edad de medicion a 17 años']

Nulos en CSV:
pais                                0
año                                 0
grupo_etario                        0
porcentaje_internet                 0
years_since_2016                    0
grupo_18 a 25 años de edad          0
grupo_26 a 50 años de edad          0
grupo_51 a 65 años                  0
grupo_66 años en adelante           0
grupo_edad de medicion a 17 años    0

Primeras filas:


,pais,año,grupo_etario,porcentaje_internet,years_since_2016,grupo_18 a 25 años de edad,grupo_26 a 50 años de edad,grupo_51 a 65 años,grupo_66 años en adelante,grupo_edad de medicion a 17 años
0,Argentina,2016,edad de medicion a 17 años,76.0,0,0,0,0,0,1
1,Argentina,2017,edad de medicion a 17 años,76.0,1,0,0,0,0,1
2,Argentina,2018,edad de medicion a 17 años,79.0,2,0,0,0,0,1
3,Argentina,2019,edad de medicion a 17 años,79.0,3,0,0,0,0,1
4,Argentina,2020,edad de medicion a 17 años,88.0,4,0,0,0,0,1
